# V10 patient-specific LOSO multi-model comparison

This notebook is the pre-specified multi-model follow-up to V9A. It compares
Random Forest, XGBoost and RBF-SVM while preserving the same leakage-controlled
patient-specific LOSO design, feature-selection rules, continuous cross-fitting
and alarm protocol.

## Locked design

- One held-out seizure per outer fold.
- Training only is down-sampled to 1:1; the held-out classification set remains
  180 preictal versus 540 interictal epochs.
- Standardisation and mutual-information selection are fitted only on the
  outer-training data.
- F1-F4 use global Top-20 MI selection.
- F5 uses 10 handcrafted + 10 PLI features.
- F6 uses 10 handcrafted + 10 wPLI features.
- F7 uses 10 handcrafted + 5 PLI + 5 wPLI features.
- The selected 20-feature representation is identical across classifiers within
  an outer fold, isolating classifier effects from feature-selection effects.
- RF uses the locked V9A parameters and provides an exact reproducibility anchor.
- XGBoost and RBF-SVM use a compact, pre-declared candidate set selected by
  grouped inner cross-validation on outer-training seizures only. MCC is the
  inner selection criterion. Held-out test and stream data never participate in
  tuning.
- Primary alarm: causal 5-min firing power, threshold 0.70, refractory 30 min.
- Refractory implementation is intentionally unchanged from V9A: alarm history
  is reset during lockout, so the algorithm also requires re-arming afterwards.
- FAR/h uses only `EligibleForFAR=1`; gaps and invalid physiological phases reset
  alarm history.
- Alarm sensitivity settings remain secondary analyses.

V10 remains an exploratory model-comparison analysis because V9A informed its
construction. It must not be described as independent external validation.


In [ ]:
# ==========================================================================
# 0. IMPORTS AND LOCKED V10 CONFIGURATION
# ==========================================================================
import gc
import json
import math
import re
import shutil
import time
import traceback
from collections import deque
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
from IPython.display import display
from scipy.stats import friedmanchisquare, wilcoxon
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import confusion_matrix, make_scorer, matthews_corrcoef
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

try:
    import xgboost
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError('V10 requires xgboost, which is normally preinstalled on Kaggle') from exc

RANDOM_SEED = 42
EPOCH_SEC = 10
SPH_SEC = 0
SOP_SEC = 30 * 60
REFRACTORY_SEC = 30 * 60
PRIMARY_ALARM_NAME = 'FP_5min_theta_0.70'
PRIMARY_FP_WINDOW_EPOCHS = 30
PRIMARY_FP_THRESHOLD = 0.70
N_SELECTED_FEATURES = 20
TRAIN_NEG_POS_RATIO = 1.0

MODEL_ORDER = ['RandomForest', 'XGBoost', 'SVM_RBF']
INNER_CV_MAX_SPLITS = 3
INNER_SCORING = 'MCC'
GRID_N_JOBS = 2

# F1-F4 retain V8's global Top-20 rule. Quotas apply only to mixed blocks.
BLOCK_BALANCED_QUOTAS = {
    ('Handcrafted', 'Graph_PLI'): {'Handcrafted': 10, 'Graph_PLI': 10},
    ('Handcrafted', 'Graph_wPLI'): {'Handcrafted': 10, 'Graph_wPLI': 10},
    ('Handcrafted', 'Graph_PLI', 'Graph_wPLI'): {
        'Handcrafted': 10, 'Graph_PLI': 5, 'Graph_wPLI': 5,
    },
}

ALARM_SENSITIVITY_SETTINGS = {
    'FP_2min_theta_0.70': {'mode': 'firing_power', 'window_epochs': 12, 'threshold': 0.70},
    PRIMARY_ALARM_NAME: {'mode': 'firing_power', 'window_epochs': 30, 'threshold': 0.70},
    'FP_10min_theta_0.70': {'mode': 'firing_power', 'window_epochs': 60, 'threshold': 0.70},
    'FP_5min_theta_0.50': {'mode': 'firing_power', 'window_epochs': 30, 'threshold': 0.50},
    'FP_5min_theta_0.80': {'mode': 'firing_power', 'window_epochs': 30, 'threshold': 0.80},
    'Legacy_90_consecutive': {'mode': 'consecutive', 'k': 90},
}
assert ALARM_SENSITIVITY_SETTINGS[PRIMARY_ALARM_NAME] == {
    'mode': 'firing_power',
    'window_epochs': PRIMARY_FP_WINDOW_EPOCHS,
    'threshold': PRIMARY_FP_THRESHOLD,
}

INPUT_SEARCH_ROOT = Path('/kaggle/input')
MODEL_READY_ROOT_OVERRIDE = None
OUTPUT_ROOT = Path('/kaggle/working/LOSO_MULTIMODEL_GAPAWARE_V10')

# First run: True. To resume after interruption without deleting completed
# subjects, change to False and rerun the notebook from the beginning.
RESET_OUTPUT = True

RF_PARAMS = {
    'n_estimators': 500,
    'max_depth': 15,
    'min_samples_leaf': 4,
    'max_features': 'sqrt',
    'class_weight': None,
    'random_state': RANDOM_SEED,
    'n_jobs': -1,
}

XGB_BASE_PARAMS = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 1,
    'reg_lambda': 1.0,
    'random_state': RANDOM_SEED,
    'tree_method': 'hist',
    'n_jobs': 1,
}

# List-of-dictionaries prevents an unintended Cartesian product and keeps the
# nested computation feasible. Each dictionary is one pre-declared candidate.
XGB_PARAM_GRID = [
    {'n_estimators': [200], 'max_depth': [3], 'learning_rate': [0.05]},
    {'n_estimators': [400], 'max_depth': [3], 'learning_rate': [0.03]},
    {'n_estimators': [300], 'max_depth': [5], 'learning_rate': [0.05]},
    {'n_estimators': [200], 'max_depth': [5], 'learning_rate': [0.10]},
]

SVM_BASE_PARAMS = {
    'kernel': 'rbf',
    'class_weight': None,
    'probability': False,
    'cache_size': 1024,
}
SVM_PARAM_GRID = [
    {'C': [0.1], 'gamma': ['scale']},
    {'C': [1.0], 'gamma': ['scale']},
    {'C': [10.0], 'gamma': ['scale']},
    {'C': [1.0], 'gamma': [0.1]},
    {'C': [10.0], 'gamma': [0.01]},
]

BLOCK_PREFIX = {
    'Handcrafted': 'HC_',
    'Graph_PLI': 'PLI_',
    'Graph_wPLI': 'wPLI_',
}


In [ ]:
# ==========================================================================
# 1. DISCOVER AND AUDIT THE MODEL-READY DATASET
# ==========================================================================
def discover_model_ready_root():
    if MODEL_READY_ROOT_OVERRIDE is not None:
        candidates = [Path(MODEL_READY_ROOT_OVERRIDE)]
    else:
        candidates = list(INPUT_SEARCH_ROOT.rglob('ModelReady_GAPAWARE_V4'))
    valid = []
    for root in candidates:
        required = [root / 'Metadata', root / 'Handcrafted', root / 'Graph_PLI',
                    root / 'Graph_wPLI', root / 'feature_set_definitions.json']
        if all(p.exists() for p in required):
            valid.append(root.resolve())
    valid = sorted(set(valid))
    if not valid:
        raise FileNotFoundError('Could not find a complete ModelReady_GAPAWARE_V4 dataset')
    if len(valid) > 1:
        raise RuntimeError(f'Multiple model-ready roots found: {valid}; set MODEL_READY_ROOT_OVERRIDE')
    return valid[0]


MODEL_ROOT = discover_model_ready_root()
with open(MODEL_ROOT / 'feature_set_definitions.json', encoding='utf-8') as file:
    FEATURE_SET_DEFINITIONS = json.load(file)

subjects = sorted(p.name for p in (MODEL_ROOT / 'Metadata').iterdir()
                  if p.is_dir() and re.fullmatch(r'chb\d{2}', p.name))
if not subjects:
    raise RuntimeError('No model-ready subjects found')

print('Model-ready root:', MODEL_ROOT)
print(f'Subjects ({len(subjects)}):', subjects)
print('Feature sets:', list(FEATURE_SET_DEFINITIONS))



In [ ]:
# ==========================================================================
# 2. LOAD ONE SUBJECT AND ALIGN ALL THREE FEATURE BLOCKS
# ==========================================================================
def is_testing_stem(stem):
    return re.search(r'_block_\d+$', stem) is not None


def keys_for_metadata(meta):
    if 'Class' in meta.columns:
        return ['Patient', 'Class', 'SeizureID', 'Epoch']
    return ['Patient', 'SeizureID', 'Block', 'Epoch']


def feature_columns_for_blocks(frame, blocks):
    prefixes = tuple(BLOCK_PREFIX[b] for b in blocks)
    cols = [c for c in frame.columns if c.startswith(prefixes)]
    if not cols:
        raise ValueError(f'No features found for blocks={blocks}')
    return cols


def load_subject(subject):
    meta_files = sorted((MODEL_ROOT / 'Metadata' / subject).glob('*.parquet'))
    if not meta_files:
        raise FileNotFoundError(f'No metadata files for {subject}')
    training, streams = [], []

    for meta_path in meta_files:
        stem = meta_path.stem
        meta = pd.read_parquet(meta_path)
        keys = keys_for_metadata(meta)
        if meta.duplicated(keys).any():
            raise AssertionError(f'{subject}/{stem}: duplicate metadata epoch keys')
        frame = meta.copy()
        for block in BLOCK_PREFIX:
            path = MODEL_ROOT / block / subject / f'{stem}.parquet'
            if not path.exists():
                raise FileNotFoundError(path)
            features = pd.read_parquet(path)
            if features.duplicated(keys).any():
                raise AssertionError(f'{block}/{subject}/{stem}: duplicate keys')
            feature_cols = [c for c in features.columns if c not in keys]
            frame = frame.merge(features[keys + feature_cols], on=keys,
                                how='inner', validate='one_to_one')
        if len(frame) != len(meta):
            raise AssertionError(f'{subject}/{stem}: feature merge lost epochs')
        (streams if is_testing_stem(stem) else training).append(frame)

    train_pool = pd.concat(training, ignore_index=True)
    stream_pool = pd.concat(streams, ignore_index=True)
    train_pool['Class'] = train_pool['Class'].map({'interictal': 0, 'preictal': 1})
    if train_pool['Class'].isna().any():
        raise ValueError(f'{subject}: unknown training class')
    train_pool['Class'] = train_pool['Class'].astype(int)
    stream_pool['EpochStart'] = pd.to_datetime(stream_pool['EpochStart'], utc=True)

    # Strong cohort invariants inherited from segmentation.
    for sid, group in train_pool.groupby('SeizureID'):
        counts = group['Class'].value_counts().to_dict()
        if counts.get(1, 0) != 180 or counts.get(0, 0) != 540:
            raise AssertionError(f'{subject}/seizure {sid}: expected 180/540, got {counts}')
        target = stream_pool.loc[stream_pool['SeizureID'] == sid]
        if int(target['TargetPreictal'].sum()) != 180:
            raise AssertionError(f'{subject}/seizure {sid}: continuous preictal != 180 epochs')
        if target.duplicated(['Epoch']).any():
            raise AssertionError(f'{subject}/seizure {sid}: duplicate continuous Epoch')
    return train_pool, stream_pool



In [ ]:
# ==========================================================================
# 3. TRAINING-ONLY BALANCING, SCALING, MI SELECTION AND INNER TUNING
# ==========================================================================
def balance_training(frame, fold_seed):
    positive = frame.loc[frame['Class'] == 1]
    negative = frame.loc[frame['Class'] == 0]
    if positive.empty or negative.empty:
        raise ValueError('Training fold must contain both classes')
    n_negative = min(len(negative), int(round(len(positive) * TRAIN_NEG_POS_RATIO)))
    negative = negative.sample(n=n_negative, replace=False, random_state=fold_seed)
    return (pd.concat([positive, negative], ignore_index=True)
              .sample(frac=1.0, random_state=fold_seed)
              .reset_index(drop=True))


def feature_block(feature_name):
    matches = [block for block, prefix in BLOCK_PREFIX.items()
               if feature_name.startswith(prefix)]
    if len(matches) != 1:
        raise ValueError(f'Cannot assign feature to exactly one block: {feature_name}')
    return matches[0]


def feature_selection_plan(blocks):
    block_set = set(blocks)
    for configured_blocks, quotas in BLOCK_BALANCED_QUOTAS.items():
        if len(blocks) == len(configured_blocks) and block_set == set(configured_blocks):
            if sum(quotas.values()) != N_SELECTED_FEATURES:
                raise AssertionError(f'Block quotas do not sum to {N_SELECTED_FEATURES}')
            return 'block_balanced_mi', dict(quotas)
    return 'global_top_mi', {'GLOBAL': N_SELECTED_FEATURES}


def select_feature_indices(feature_cols, mi_scores, blocks):
    mode, plan = feature_selection_plan(blocks)
    scores = np.asarray(mi_scores, dtype=float)
    if len(scores) != len(feature_cols):
        raise ValueError('MI-score/feature length mismatch')

    if mode == 'global_top_mi':
        k = min(N_SELECTED_FEATURES, len(feature_cols))
        selected_idx = np.sort(np.argsort(scores, kind='mergesort')[-k:])
        ranked = sorted(selected_idx, key=lambda i: (-scores[i], feature_cols[i]))
        records = []
        for rank, idx in enumerate(ranked, start=1):
            records.append({
                'Feature': feature_cols[idx], 'MIScore': float(scores[idx]),
                'FeatureBlock': feature_block(feature_cols[idx]),
                'RankWithinBlock': np.nan, 'RequestedBlockQuota': np.nan,
                'RankWithinSelected': rank,
            })
        return np.asarray(selected_idx, dtype=np.int64), records, mode, plan

    selected_idx, records = [], []
    for block in blocks:
        quota = int(plan[block])
        candidates = [i for i, name in enumerate(feature_cols)
                      if feature_block(name) == block]
        if len(candidates) < quota:
            raise ValueError(f'{block}: requested {quota} features, found {len(candidates)}')
        local = np.argsort(scores[candidates], kind='mergesort')[-quota:]
        chosen = [candidates[i] for i in local]
        ranked = sorted(chosen, key=lambda i: (-scores[i], feature_cols[i]))
        for block_rank, idx in enumerate(ranked, start=1):
            selected_idx.append(idx)
            records.append({
                'Feature': feature_cols[idx], 'MIScore': float(scores[idx]),
                'FeatureBlock': block, 'RankWithinBlock': block_rank,
                'RequestedBlockQuota': quota,
                'RankWithinSelected': len(records) + 1,
            })

    if len(selected_idx) != N_SELECTED_FEATURES or len(set(selected_idx)) != len(selected_idx):
        raise AssertionError('Block-balanced selector did not return 20 unique features')
    return (np.sort(np.asarray(selected_idx, dtype=np.int64)), records,
            mode, plan)


def prepare_fold(train, test, stream, feature_cols, blocks, fold_seed):
    balanced = balance_training(train, fold_seed)
    X_train = balanced[feature_cols].to_numpy(dtype=np.float32)
    y_train = balanced['Class'].to_numpy(dtype=np.int8)
    groups_train = balanced['SeizureID'].to_numpy(dtype=np.int32)
    X_test = test[feature_cols].to_numpy(dtype=np.float32)
    y_test = test['Class'].to_numpy(dtype=np.int8)
    X_stream = stream[feature_cols].to_numpy(dtype=np.float32)

    for name, array in [('train', X_train), ('test', X_test), ('stream', X_stream)]:
        if not np.isfinite(array).all():
            raise ValueError(f'{name}: NaN or Inf before modelling')

    scaler = StandardScaler()
    X_train_z = scaler.fit_transform(X_train)
    X_test_z = scaler.transform(X_test)
    X_stream_z = scaler.transform(X_stream)

    mi_scores = mutual_info_classif(X_train_z, y_train, random_state=RANDOM_SEED)
    selected_idx, selected, selection_mode, selection_plan = select_feature_indices(
        feature_cols, mi_scores, blocks)
    return {
        'X_train': X_train_z[:, selected_idx], 'y_train': y_train,
        'groups_train': groups_train,
        'X_test': X_test_z[:, selected_idx], 'y_test': y_test,
        'X_stream': X_stream_z[:, selected_idx],
        'selected': selected, 'balanced_n': len(balanced),
        'scaler': scaler, 'selected_idx': selected_idx,
        'selection_mode': selection_mode, 'selection_plan': selection_plan,
    }


def _jsonable_params(params):
    clean = {}
    for key, value in params.items():
        clean[key] = value.item() if isinstance(value, np.generic) else value
    return clean


def fit_classifier(model_name, X, y, groups, fold_seed):
    start = time.perf_counter()
    if model_name == 'RandomForest':
        model = RandomForestClassifier(**RF_PARAMS)
        model.fit(X, y)
        summary = {
            'TuningMode': 'locked_V9A_parameters', 'InnerSplits': 0,
            'BestInnerMCC': np.nan, 'BestInnerMCC_SD': np.nan,
            'BestParams': json.dumps(_jsonable_params(RF_PARAMS), sort_keys=True),
        }
        candidates = []
    else:
        unique_groups = np.unique(groups)
        n_splits = min(INNER_CV_MAX_SPLITS, len(unique_groups))
        if n_splits < 2:
            raise ValueError('Grouped inner CV requires at least two training seizures')
        cv = StratifiedGroupKFold(
            n_splits=n_splits, shuffle=True, random_state=fold_seed)
        scorer = make_scorer(matthews_corrcoef)
        if model_name == 'XGBoost':
            estimator = XGBClassifier(**XGB_BASE_PARAMS)
            grid = XGB_PARAM_GRID
        elif model_name == 'SVM_RBF':
            estimator = SVC(**SVM_BASE_PARAMS)
            grid = SVM_PARAM_GRID
        else:
            raise ValueError(f'Unknown model: {model_name}')
        search = GridSearchCV(
            estimator=estimator, param_grid=grid, scoring=scorer, cv=cv,
            refit=True, n_jobs=GRID_N_JOBS, error_score='raise',
            return_train_score=False)
        search.fit(X, y, groups=groups)
        model = search.best_estimator_
        best_index = int(search.best_index_)
        summary = {
            'TuningMode': 'grouped_inner_CV', 'InnerSplits': n_splits,
            'BestInnerMCC': float(search.cv_results_['mean_test_score'][best_index]),
            'BestInnerMCC_SD': float(search.cv_results_['std_test_score'][best_index]),
            'BestParams': json.dumps(_jsonable_params(search.best_params_), sort_keys=True),
        }
        candidates = []
        for idx, params in enumerate(search.cv_results_['params']):
            candidates.append({
                'CandidateIndex': idx,
                'Params': json.dumps(_jsonable_params(params), sort_keys=True),
                'MeanInnerMCC': float(search.cv_results_['mean_test_score'][idx]),
                'SDInnerMCC': float(search.cv_results_['std_test_score'][idx]),
                'RankInnerMCC': int(search.cv_results_['rank_test_score'][idx]),
                'Selected': int(idx == best_index),
            })
    summary['FitAndTuneTime_sec'] = time.perf_counter() - start
    return model, summary, candidates


def epoch_fingerprints(frame, fingerprint_cols):
    return pd.util.hash_pandas_object(
        frame[fingerprint_cols], index=False, categorize=False
    ).to_numpy(dtype=np.uint64)


def build_interictal_owner_map(train_pool, fingerprint_cols):
    negative = train_pool.loc[train_pool['Class'] == 0]
    hashes = epoch_fingerprints(negative, fingerprint_cols)
    owners = negative['SeizureID'].to_numpy(dtype=int)
    mapping = pd.DataFrame({'Hash': hashes, 'Owner': owners})
    conflicts = mapping.groupby('Hash')['Owner'].nunique()
    if (conflicts > 1).any():
        raise AssertionError('An interictal fingerprint belongs to multiple LOSO folds')
    return mapping.drop_duplicates('Hash').set_index('Hash')['Owner'].to_dict()


def predict_pipeline(frame, feature_cols, fitted):
    scaler, selected_idx, model = fitted
    values = frame[feature_cols].to_numpy(dtype=np.float32)
    if not np.isfinite(values).all():
        raise ValueError('NaN or Inf during cross-fitted stream prediction')
    transformed = scaler.transform(values)[:, selected_idx]
    return model.predict(transformed).astype(np.int8)


In [ ]:
# ==========================================================================
# 4. CAUSAL CONTINUOUS ALARM, FAR/H AND TIME-TO-WARNING
# ==========================================================================
def _validate_alarm_setting(setting):
    mode = setting.get('mode')
    if mode == 'firing_power':
        window = int(setting['window_epochs'])
        threshold = float(setting['threshold'])
        if window < 1 or not (0 < threshold <= 1):
            raise ValueError(f'Invalid firing-power setting: {setting}')
        return mode, window, threshold, math.ceil(window * threshold)
    if mode == 'consecutive':
        k = int(setting['k'])
        if k < 1:
            raise ValueError(f'Invalid consecutive setting: {setting}')
        return mode, k, 1.0, k
    raise ValueError(f'Unknown alarm mode: {mode}')


def run_causal_alarm(y_pred, stream, setting):
    stream = stream.reset_index(drop=True)
    predictions = np.asarray(y_pred, dtype=np.int8)
    if len(predictions) != len(stream):
        raise ValueError('Prediction/stream length mismatch')
    if not np.isin(predictions, [0, 1]).all():
        raise ValueError('Alarm input predictions must be binary')

    mode, window_epochs, threshold, required_positive = _validate_alarm_setting(setting)

    history = deque(maxlen=window_epochs)
    refractory_until = None
    previous_start = None
    alarms = []

    for idx, (pred, row) in enumerate(zip(predictions, stream.itertuples(index=False))):
        start = row.EpochStart
        end = start + pd.Timedelta(seconds=EPOCH_SEC)
        gap = bool(row.Discontinuity)
        if previous_start is not None:
            elapsed = (start - previous_start).total_seconds()
            gap = gap or elapsed > EPOCH_SEC + 0.5 or elapsed <= 0
        previous_start = start

        phase = str(row.Phase).lower()
        valid_state = phase in {'interictal', 'preictal'}
        if gap or not valid_state:
            history.clear()
            continue
        if refractory_until is not None and end < refractory_until:
            history.clear()
            continue

        if mode == 'consecutive':
            if pred == 0:
                history.clear()
            else:
                history.append(1)
        else:
            history.append(int(pred))

        full_window = len(history) == window_epochs
        positive_count = int(sum(history))
        if full_window and positive_count >= required_positive:
            alarms.append({
                'Index': idx,
                'AlarmTime': end,
                'Phase': phase,
                'EligibleForFAR': int(row.EligibleForFAR),
                'TargetPreictal': int(row.TargetPreictal),
                'WindowEpochs': window_epochs,
                'PositiveEpochs': positive_count,
                'FiringPower': positive_count / window_epochs,
            })
            history.clear()
            refractory_until = end + pd.Timedelta(seconds=REFRACTORY_SEC)
    return alarms


def evaluate_alarm(y_pred, stream, setting):
    stream = stream.sort_values('Epoch', kind='stable').reset_index(drop=True)
    target = stream.loc[stream['TargetPreictal'].astype(bool)]
    if len(target) != 180:
        raise AssertionError(f'Expected 180 target-preictal epochs, got {len(target)}')
    seizure_onset = target['EpochStart'].max() + pd.Timedelta(seconds=EPOCH_SEC)
    valid_start = seizure_onset - pd.Timedelta(seconds=SPH_SEC + SOP_SEC)
    valid_end = seizure_onset - pd.Timedelta(seconds=SPH_SEC)

    alarms = run_causal_alarm(y_pred, stream, setting)
    true_alarms = [a for a in alarms if valid_start <= a['AlarmTime'] < valid_end]
    late_alarms = [a for a in alarms if valid_end <= a['AlarmTime'] < seizure_onset]
    false_alarms = [a for a in alarms if a['EligibleForFAR'] == 1]
    eligible_hours = float(stream['EligibleForFAR'].astype(bool).sum() * EPOCH_SEC / 3600.0)
    far_h = len(false_alarms) / eligible_hours if eligible_hours > 0 else np.nan
    ttw = ((seizure_onset - true_alarms[0]['AlarmTime']).total_seconds() / 60.0
           if true_alarms else np.nan)
    return {
        'SeizurePredicted': int(bool(true_alarms)),
        'FalseAlarms': len(false_alarms),
        'LateAlarms': len(late_alarms),
        'EligibleInterictalHours': eligible_hours,
        'FAR_per_h': far_h,
        'TimeToWarning_min': ttw,
        'TotalAlarms': len(alarms),
    }


def run_alarm_unit_tests():
    onset = pd.Timestamp('2076-01-01T01:00:00Z')
    starts = pd.date_range(onset - pd.Timedelta(minutes=40), periods=240, freq='10s')
    stream = pd.DataFrame({
        'Epoch': np.arange(240), 'EpochStart': starts,
        'Phase': ['interictal'] * 60 + ['preictal'] * 180,
        'TargetPreictal': [0] * 60 + [1] * 180,
        'EligibleForFAR': [1] * 60 + [0] * 180,
        'Discontinuity': 0,
    })
    predictions = np.array([0] * 60 + [1] * 180)
    primary = ALARM_SENSITIVITY_SETTINGS[PRIMARY_ALARM_NAME]
    result = evaluate_alarm(predictions, stream, primary)
    assert result['SeizurePredicted'] == 1
    # The 5-min causal window already contains nine preceding interictal
    # negatives, so the threshold is crossed after 21 positive epochs (3.5 min).
    assert abs(result['TimeToWarning_min'] - 26.5) < 1e-9
    assert result['FalseAlarms'] == 0

    # Nine isolated negatives are tolerated: 21/30 still reaches theta=0.70.
    noisy = predictions.copy()
    noisy[61:70] = 0
    result = evaluate_alarm(noisy, stream, primary)
    assert result['SeizurePredicted'] == 1
    assert abs(result['TimeToWarning_min'] - 25.0) < 1e-9

    # A discontinuity clears all prior evidence. With only 20 valid epochs
    # after the gap, no full 30-epoch firing-power window can be evaluated.
    broken = stream.copy()
    short_predictions = np.zeros_like(predictions)
    short_predictions[60:90] = 1
    broken.loc[70, 'Discontinuity'] = 1
    result = evaluate_alarm(short_predictions, broken, primary)
    assert result['SeizurePredicted'] == 0

    # Legacy rule remains reproducible only as a sensitivity analysis.
    result = evaluate_alarm(
        predictions, stream, ALARM_SENSITIVITY_SETTINGS['Legacy_90_consecutive'])
    assert result['SeizurePredicted'] == 1
    assert abs(result['TimeToWarning_min'] - 15.0) < 1e-9
    print('Causal alarm unit tests: PASS')


run_alarm_unit_tests()



In [ ]:
# ==========================================================================
# 5. RUN ONE SUBJECT ACROSS F1-F7 AND THREE CLASSIFIERS
# ==========================================================================
def epoch_counts(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return int(tn), int(fp), int(fn), int(tp)


def run_subject(subject):
    train_pool, stream_pool = load_subject(subject)
    seizure_ids = sorted(train_pool.loc[train_pool['Class'] == 1, 'SeizureID'].unique())
    fingerprint_cols = feature_columns_for_blocks(train_pool, ['Handcrafted'])
    owner_map = build_interictal_owner_map(train_pool, fingerprint_cols)

    streams, owner_arrays, crossfit_qc = {}, {}, {}
    for sid in seizure_ids:
        stream = (stream_pool.loc[stream_pool['SeizureID'] == sid]
                  .sort_values('Epoch', kind='stable').reset_index(drop=True))
        hashes = epoch_fingerprints(stream, fingerprint_cols)
        eligible = stream['EligibleForFAR'].astype(bool).to_numpy()
        owners = np.full(len(stream), -1, dtype=np.int32)
        if eligible.any():
            owners[eligible] = np.array(
                [owner_map.get(int(h), -1) for h in hashes[eligible]],
                dtype=np.int32)
        matched = int((eligible & (owners >= 0)).sum())
        total_eligible = int(eligible.sum())
        streams[int(sid)] = stream
        owner_arrays[int(sid)] = owners
        crossfit_qc[int(sid)] = {
            'EligibleInterictalEpochs': total_eligible,
            'CrossFitMatchedInterictalEpochs': matched,
            'CrossFitUnmatchedInterictalEpochs': total_eligible - matched,
            'CrossFitMatchFraction': (matched / total_eligible
                                      if total_eligible else np.nan),
        }
        print(f'{subject} | seizure {sid} | cross-fit matched {matched}/'
              f'{total_eligible} eligible interictal epochs', flush=True)

    fold_rows, selected_rows = [], []
    alarm_sensitivity_rows, tuning_rows, tuning_candidate_rows = [], [], []

    for feature_set, blocks in FEATURE_SET_DEFINITIONS.items():
        feature_cols = feature_columns_for_blocks(train_pool, blocks)
        print(f'{subject} | {feature_set} | {len(feature_cols)} raw features', flush=True)
        caches = {model_name: {} for model_name in MODEL_ORDER}

        # Fit every outer LOSO model before cross-fitted stream prediction.
        for fold_index, sid in enumerate(seizure_ids):
            sid = int(sid)
            train = train_pool.loc[train_pool['SeizureID'] != sid]
            test = (train_pool.loc[train_pool['SeizureID'] == sid]
                    .sort_values(['Class', 'Epoch'], kind='stable'))
            stream = streams[sid]
            fold_seed = RANDOM_SEED + fold_index
            prepared = prepare_fold(
                train, test, stream, feature_cols, blocks, fold_seed)

            for selected_record in prepared['selected']:
                selected_rows.append({
                    'Subject': subject, 'FeatureSet': feature_set,
                    'HeldOutSeizure': sid,
                    'FeatureSelectionMode': prepared['selection_mode'],
                    'FeatureSelectionPlan': json.dumps(
                        prepared['selection_plan'], sort_keys=True),
                    **selected_record,
                })

            for model_name in MODEL_ORDER:
                print(f'  fold {sid} | {model_name}', flush=True)
                model, tuning, candidates = fit_classifier(
                    model_name, prepared['X_train'], prepared['y_train'],
                    prepared['groups_train'], fold_seed)
                inference_start = time.perf_counter()
                y_pred = model.predict(prepared['X_test']).astype(np.int8)
                test_inference_sec = time.perf_counter() - inference_start
                inference_start = time.perf_counter()
                default_stream_pred = model.predict(
                    prepared['X_stream']).astype(np.int8)
                stream_inference_sec = time.perf_counter() - inference_start
                caches[model_name][sid] = {
                    'fitted': (prepared['scaler'], prepared['selected_idx'], model),
                    'y_test': prepared['y_test'], 'y_pred': y_pred,
                    'default_stream_pred': default_stream_pred,
                    'test_inference_sec': test_inference_sec,
                    'stream_inference_sec': stream_inference_sec,
                    'balanced_n': prepared['balanced_n'], 'train_n': len(train),
                    'selection_mode': prepared['selection_mode'],
                    'selection_plan': prepared['selection_plan'],
                }
                tuning_rows.append({
                    'Subject': subject, 'FeatureSet': feature_set,
                    'Model': model_name, 'HeldOutSeizure': sid, **tuning,
                })
                for candidate in candidates:
                    tuning_candidate_rows.append({
                        'Subject': subject, 'FeatureSet': feature_set,
                        'Model': model_name, 'HeldOutSeizure': sid,
                        'InnerSplits': tuning['InnerSplits'], **candidate,
                    })

        for model_name in MODEL_ORDER:
            cache = caches[model_name]
            for sid in map(int, seizure_ids):
                stream = streams[sid]
                owners = owner_arrays[sid]
                eligible = stream['EligibleForFAR'].astype(bool).to_numpy()
                result = cache[sid]
                y_stream = result['default_stream_pred'].copy()

                for owner in sorted(set(owners[eligible]) - {-1}):
                    idx = np.flatnonzero(eligible & (owners == owner))
                    if int(owner) not in cache:
                        raise AssertionError(f'Missing cross-fit model for owner {owner}')
                    y_stream[idx] = predict_pipeline(
                        stream.iloc[idx], feature_cols, cache[int(owner)]['fitted'])

                y_test, y_pred = result['y_test'], result['y_pred']
                tn, fp, fn, tp = epoch_counts(y_test, y_pred)
                primary_setting = ALARM_SENSITIVITY_SETTINGS[PRIMARY_ALARM_NAME]
                alarm = evaluate_alarm(y_stream, stream, primary_setting)
                qc = crossfit_qc[sid]

                fold_rows.append({
                    'Subject': subject, 'FeatureSet': feature_set,
                    'Model': model_name, 'HeldOutSeizure': sid,
                    'AlarmRule': PRIMARY_ALARM_NAME,
                    'AlarmMode': primary_setting['mode'],
                    'FiringPowerWindow_epochs': primary_setting['window_epochs'],
                    'FiringPowerWindow_min': (
                        primary_setting['window_epochs'] * EPOCH_SEC / 60.0),
                    'FiringPowerThreshold': primary_setting['threshold'],
                    'RequiredPositiveEpochs': math.ceil(
                        primary_setting['window_epochs'] * primary_setting['threshold']),
                    'SPH_min': SPH_SEC / 60.0, 'SOP_min': SOP_SEC / 60.0,
                    'Refractory_min': REFRACTORY_SEC / 60.0,
                    'TrainEpochsBeforeBalance': result['train_n'],
                    'TrainEpochsAfterBalance': result['balanced_n'],
                    'TestInferenceTime_sec': result['test_inference_sec'],
                    'TestInference_ms_per_epoch': (
                        1000.0 * result['test_inference_sec'] / len(y_test)),
                    'StreamInferenceTime_sec': result['stream_inference_sec'],
                    'StreamInference_ms_per_epoch': (
                        1000.0 * result['stream_inference_sec'] / len(stream)),
                    'FeatureSelectionMode': result['selection_mode'],
                    'FeatureSelectionPlan': json.dumps(
                        result['selection_plan'], sort_keys=True),
                    'TestPreictal': int((y_test == 1).sum()),
                    'TestInterictal': int((y_test == 0).sum()),
                    'StreamEpochs': len(stream), **qc,
                    'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp, **alarm,
                })

                for alarm_name, alarm_setting in ALARM_SENSITIVITY_SETTINGS.items():
                    sensitivity_alarm = evaluate_alarm(y_stream, stream, alarm_setting)
                    mode, window_epochs, threshold, required = _validate_alarm_setting(
                        alarm_setting)
                    alarm_sensitivity_rows.append({
                        'Subject': subject, 'FeatureSet': feature_set,
                        'Model': model_name, 'HeldOutSeizure': sid,
                        'AlarmRule': alarm_name,
                        'IsPrimary': int(alarm_name == PRIMARY_ALARM_NAME),
                        'AlarmMode': mode, 'WindowEpochs': window_epochs,
                        'Window_min': window_epochs * EPOCH_SEC / 60.0,
                        'Threshold': threshold, 'RequiredPositiveEpochs': required,
                        'SPH_min': SPH_SEC / 60.0, 'SOP_min': SOP_SEC / 60.0,
                        'Refractory_min': REFRACTORY_SEC / 60.0, **qc,
                        **sensitivity_alarm,
                    })

        del caches
        gc.collect()

    del train_pool, stream_pool, streams, owner_arrays
    gc.collect()
    return (fold_rows, selected_rows, alarm_sensitivity_rows,
            tuning_rows, tuning_candidate_rows)


In [ ]:
# ==========================================================================
# 6. COMPLETE-COHORT EXECUTION WITH RESUMABLE SUBJECT CHECKPOINTS
# ==========================================================================
if RESET_OUTPUT and OUTPUT_ROOT.exists():
    print('[RESET OUTPUT]', OUTPUT_ROOT)
    shutil.rmtree(OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_ROOT / 'run_config.json', 'w', encoding='utf-8') as file:
    json.dump({
        'analysis_name': 'V10_LOSO_multimodel_grouped_inner_CV',
        'analysis_status': 'exploratory_follow_up_to_V9A',
        'split_design': 'epoch_disjoint_LOSO_cross_fitted_continuous_stream',
        'crossfit_fingerprint_block': 'all_900_handcrafted_features_float32',
        'random_seed': RANDOM_SEED, 'epoch_sec': EPOCH_SEC,
        'SPH_sec': SPH_SEC, 'SOP_sec': SOP_SEC,
        'refractory_sec': REFRACTORY_SEC,
        'refractory_history_policy': 'clear_during_lockout_then_rearm',
        'primary_alarm_name': PRIMARY_ALARM_NAME,
        'primary_firing_power_window_epochs': PRIMARY_FP_WINDOW_EPOCHS,
        'primary_firing_power_threshold': PRIMARY_FP_THRESHOLD,
        'alarm_sensitivity_settings': ALARM_SENSITIVITY_SETTINGS,
        'selection_rule': 'global_top20_for_F1_F4; block_balanced_top20_for_F5_F7',
        'top_mi_features': N_SELECTED_FEATURES,
        'block_balanced_quotas': [
            {'blocks': list(blocks), 'quotas': quotas}
            for blocks, quotas in BLOCK_BALANCED_QUOTAS.items()
        ],
        'train_negative_positive_ratio': TRAIN_NEG_POS_RATIO,
        'models': MODEL_ORDER, 'inner_scoring': INNER_SCORING,
        'inner_cv_max_splits': INNER_CV_MAX_SPLITS,
        'grid_n_jobs': GRID_N_JOBS,
        'software_versions': {
            'numpy': np.__version__, 'pandas': pd.__version__,
            'scipy': scipy.__version__, 'scikit_learn': sklearn.__version__,
            'xgboost': xgboost.__version__,
        },
        'inference_timing_scope': 'classifier_predict_on_precomputed_selected_features',
        'rf_params': RF_PARAMS,
        'xgb_base_params': XGB_BASE_PARAMS,
        'xgb_param_grid': XGB_PARAM_GRID,
        'svm_base_params': SVM_BASE_PARAMS,
        'svm_param_grid': SVM_PARAM_GRID,
        'feature_sets': FEATURE_SET_DEFINITIONS,
    }, file, indent=2)


def existing_records(filename):
    path = OUTPUT_ROOT / filename
    return pd.read_csv(path).to_dict('records') if path.exists() else []


all_folds = existing_records('fold_metrics.csv')
all_selected = existing_records('selected_features.csv')
all_alarm_sensitivity = existing_records('alarm_sensitivity_folds.csv')
all_tuning = existing_records('tuning_summary.csv')
all_tuning_candidates = existing_records('tuning_candidates.csv')
run_rows = existing_records('run_manifest.csv')
completed = {r['Subject'] for r in run_rows if r.get('Status') == 'completed'}

for subject in subjects:
    if subject in completed:
        print('[RESUME SKIP]', subject)
        continue
    try:
        outputs = run_subject(subject)
        fold_rows, selected_rows, alarm_rows, tuning_rows, candidate_rows = outputs
        all_folds.extend(fold_rows)
        all_selected.extend(selected_rows)
        all_alarm_sensitivity.extend(alarm_rows)
        all_tuning.extend(tuning_rows)
        all_tuning_candidates.extend(candidate_rows)
        run_rows.append({'Subject': subject, 'Status': 'completed', 'Error': ''})
    except Exception as exc:
        traceback.print_exc()
        run_rows.append({'Subject': subject, 'Status': 'failed', 'Error': repr(exc)})

    pd.DataFrame(all_folds).to_csv(OUTPUT_ROOT / 'fold_metrics.csv', index=False)
    pd.DataFrame(all_selected).to_csv(OUTPUT_ROOT / 'selected_features.csv', index=False)
    pd.DataFrame(all_alarm_sensitivity).to_csv(
        OUTPUT_ROOT / 'alarm_sensitivity_folds.csv', index=False)
    pd.DataFrame(all_tuning).to_csv(OUTPUT_ROOT / 'tuning_summary.csv', index=False)
    pd.DataFrame(all_tuning_candidates).to_csv(
        OUTPUT_ROOT / 'tuning_candidates.csv', index=False)
    (pd.DataFrame(run_rows).drop_duplicates('Subject', keep='last')
       .to_csv(OUTPUT_ROOT / 'run_manifest.csv', index=False))

run_manifest = pd.read_csv(OUTPUT_ROOT / 'run_manifest.csv')
display(run_manifest)
if (run_manifest['Status'] != 'completed').any():
    raise RuntimeError('At least one subject failed; set RESET_OUTPUT=False and rerun after inspection')


In [ ]:
# ==========================================================================
# 7. MULTI-MODEL SUMMARIES, QC AND PAIRED STATISTICAL TESTS
# ==========================================================================
def safe_div(a, b):
    return a / b if b else np.nan


def mcc_from_counts(tn, fp, fn, tp):
    den = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return ((tp * tn - fp * fn) / den) if den else np.nan


def holm_adjust(frame, p_col='PValue_Uncorrected'):
    result = frame.copy()
    if result.empty:
        result['PValue_Holm'] = []
        return result
    order = np.argsort(result[p_col].to_numpy(dtype=float))
    raw = result[p_col].to_numpy(dtype=float)[order]
    adjusted_sorted = np.empty_like(raw)
    running = 0.0
    for i, pvalue in enumerate(raw):
        running = max(running, (len(raw) - i) * pvalue)
        adjusted_sorted[i] = min(1.0, running)
    adjusted = np.empty_like(adjusted_sorted)
    adjusted[order] = adjusted_sorted
    result['PValue_Holm'] = adjusted
    return result


folds = pd.read_csv(OUTPUT_ROOT / 'fold_metrics.csv')
selected_features = pd.read_csv(OUTPUT_ROOT / 'selected_features.csv')

# Selection is shared by models and therefore must occur once per outer fold.
selection_keys = ['Subject', 'FeatureSet', 'HeldOutSeizure']
selected_totals = selected_features.groupby(selection_keys).size()
if not (selected_totals == N_SELECTED_FEATURES).all():
    raise AssertionError('At least one fold did not select exactly 20 features')
for feature_set, blocks in FEATURE_SET_DEFINITIONS.items():
    mode, plan = feature_selection_plan(blocks)
    subset = selected_features.loc[selected_features['FeatureSet'] == feature_set]
    if set(subset['FeatureSelectionMode'].dropna().unique()) != {mode}:
        raise AssertionError(f'{feature_set}: feature-selection mode mismatch')
    if mode == 'block_balanced_mi':
        counts = (subset.groupby(selection_keys + ['FeatureBlock']).size()
                  .unstack(fill_value=0))
        for block, quota in plan.items():
            if block not in counts or not (counts[block] == quota).all():
                raise AssertionError(f'{feature_set}/{block}: quota failed')
print('Feature-selection quota QC: PASS')

expected_folds = 0
for subject in subjects:
    meta = pd.concat([pd.read_parquet(p) for p in
                      (MODEL_ROOT / 'Metadata' / subject).glob('*.parquet')
                      if not is_testing_stem(p.stem)], ignore_index=True)
    expected_folds += meta.loc[meta['Class'] == 'preictal', 'SeizureID'].nunique()
expected_rows = expected_folds * len(FEATURE_SET_DEFINITIONS) * len(MODEL_ORDER)
if len(folds) != expected_rows:
    raise AssertionError(f'Expected {expected_rows} fold rows, got {len(folds)}')
if not ((folds['TestPreictal'] == 180) & (folds['TestInterictal'] == 540)).all():
    raise AssertionError('A held-out test is not 180 preictal + 540 interictal')
if set(folds['Model'].unique()) != set(MODEL_ORDER):
    raise AssertionError('Model set is incomplete')

crossfit_cols = [
    'EligibleInterictalEpochs', 'CrossFitMatchedInterictalEpochs',
    'CrossFitUnmatchedInterictalEpochs', 'CrossFitMatchFraction']
consistency = folds.groupby(['Subject', 'HeldOutSeizure'])[crossfit_cols].nunique(
    dropna=False)
if (consistency > 1).any().any():
    raise AssertionError('Cross-fit ownership changed between models/features')
crossfit_qc = (folds[[
    'Subject', 'HeldOutSeizure', 'StreamEpochs', *crossfit_cols,
    'EligibleInterictalHours']]
    .drop_duplicates(['Subject', 'HeldOutSeizure'])
    .sort_values(['Subject', 'HeldOutSeizure'], kind='stable')
    .reset_index(drop=True))
if not np.allclose(
        crossfit_qc['EligibleInterictalEpochs'] * EPOCH_SEC / 3600.0,
        crossfit_qc['EligibleInterictalHours']):
    raise AssertionError('FAR denominator lost monitoring time')
crossfit_qc.to_csv(OUTPUT_ROOT / 'crossfit_stream_qc.csv', index=False)

subject_rows = []
for (subject, model_name, feature_set), group in folds.groupby(
        ['Subject', 'Model', 'FeatureSet']):
    tn, fp, fn, tp = (int(group[c].sum()) for c in ['TN', 'FP', 'FN', 'TP'])
    sensitivity = safe_div(tp, tp + fn)
    specificity = safe_div(tn, tn + fp)
    subject_rows.append({
        'Subject': subject, 'Model': model_name, 'FeatureSet': feature_set,
        'Seizures': len(group), 'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp,
        'Accuracy': safe_div(tp + tn, tp + tn + fp + fn),
        'Sensitivity': sensitivity, 'Specificity': specificity,
        'BalancedAccuracy': np.nanmean([sensitivity, specificity]),
        'F1': safe_div(2 * tp, 2 * tp + fp + fn),
        'MCC': mcc_from_counts(tn, fp, fn, tp),
        'SeizureSensitivity': group['SeizurePredicted'].mean(),
        'FalseAlarms': int(group['FalseAlarms'].sum()),
        'EligibleInterictalHours': group['EligibleInterictalHours'].sum(),
        'FAR_per_h': safe_div(group['FalseAlarms'].sum(),
                              group['EligibleInterictalHours'].sum()),
        'MeanTimeToWarning_min': group.loc[
            group['SeizurePredicted'] == 1, 'TimeToWarning_min'].mean(),
        'LateAlarms': int(group['LateAlarms'].sum()),
    })
subject_metrics = pd.DataFrame(subject_rows)
subject_metrics.to_csv(OUTPUT_ROOT / 'subject_metrics.csv', index=False)

metric_cols = ['Accuracy', 'Sensitivity', 'Specificity', 'BalancedAccuracy',
               'F1', 'MCC', 'SeizureSensitivity', 'FAR_per_h',
               'MeanTimeToWarning_min']
summary_rows = []
for (model_name, feature_set), group in subject_metrics.groupby(['Model', 'FeatureSet']):
    for metric in metric_cols:
        summary_rows.append({
            'Model': model_name, 'FeatureSet': feature_set, 'Metric': metric,
            'Mean': group[metric].mean(), 'STD': group[metric].std(ddof=1),
            'Median': group[metric].median(), 'N': group[metric].notna().sum(),
        })
cohort_summary = pd.DataFrame(summary_rows)
cohort_summary.to_csv(OUTPUT_ROOT / 'cohort_summary.csv', index=False)

pooled_rows = []
for (model_name, feature_set), group in folds.groupby(['Model', 'FeatureSet']):
    tn, fp, fn, tp = (int(group[c].sum()) for c in ['TN', 'FP', 'FN', 'TP'])
    sensitivity, specificity = safe_div(tp, tp + fn), safe_div(tn, tn + fp)
    pooled_rows.append({
        'Model': model_name, 'FeatureSet': feature_set,
        'Subjects': int(group['Subject'].nunique()), 'Seizures': len(group),
        'Accuracy': safe_div(tp + tn, tp + tn + fp + fn),
        'Sensitivity': sensitivity, 'Specificity': specificity,
        'BalancedAccuracy': np.nanmean([sensitivity, specificity]),
        'F1': safe_div(2 * tp, 2 * tp + fp + fn),
        'MCC': mcc_from_counts(tn, fp, fn, tp),
        'SeizuresPredicted': int(group['SeizurePredicted'].sum()),
        'SeizureSensitivity': group['SeizurePredicted'].mean(),
        'FalseAlarms': int(group['FalseAlarms'].sum()),
        'EligibleInterictalHours': group['EligibleInterictalHours'].sum(),
        'FAR_per_h': safe_div(group['FalseAlarms'].sum(),
                              group['EligibleInterictalHours'].sum()),
        'MeanTimeToWarning_min': group.loc[
            group['SeizurePredicted'] == 1, 'TimeToWarning_min'].mean(),
    })
cohort_pooled_summary = pd.DataFrame(pooled_rows)
cohort_pooled_summary.to_csv(OUTPUT_ROOT / 'cohort_pooled_summary.csv', index=False)

# Secondary alarm sensitivity summaries.
alarm_folds = pd.read_csv(OUTPUT_ROOT / 'alarm_sensitivity_folds.csv')
alarm_subject_rows = []
for (subject, model_name, feature_set, alarm_rule), group in alarm_folds.groupby(
        ['Subject', 'Model', 'FeatureSet', 'AlarmRule']):
    alarm_subject_rows.append({
        'Subject': subject, 'Model': model_name, 'FeatureSet': feature_set,
        'AlarmRule': alarm_rule, 'IsPrimary': int(group['IsPrimary'].iloc[0]),
        'Seizures': len(group),
        'SeizuresPredicted': int(group['SeizurePredicted'].sum()),
        'SeizureSensitivity': group['SeizurePredicted'].mean(),
        'FalseAlarms': int(group['FalseAlarms'].sum()),
        'EligibleInterictalHours': group['EligibleInterictalHours'].sum(),
        'FAR_per_h': safe_div(group['FalseAlarms'].sum(),
                              group['EligibleInterictalHours'].sum()),
        'MeanTimeToWarning_min': group.loc[
            group['SeizurePredicted'] == 1, 'TimeToWarning_min'].mean(),
    })
alarm_subject_metrics = pd.DataFrame(alarm_subject_rows)
alarm_subject_metrics.to_csv(
    OUTPUT_ROOT / 'alarm_sensitivity_subject_metrics.csv', index=False)

alarm_cohort_summary = (alarm_subject_metrics
    .groupby(['Model', 'FeatureSet', 'AlarmRule', 'IsPrimary'], as_index=False)
    .agg(Subjects=('Subject', 'nunique'),
         MeanSeizureSensitivity=('SeizureSensitivity', 'mean'),
         SDSeizureSensitivity=('SeizureSensitivity', 'std'),
         MeanFAR_per_h=('FAR_per_h', 'mean'),
         SDFAR_per_h=('FAR_per_h', 'std'),
         MeanTimeToWarning_min=('MeanTimeToWarning_min', 'mean')))
alarm_cohort_summary.to_csv(
    OUTPUT_ROOT / 'alarm_sensitivity_cohort_summary.csv', index=False)

alarm_pooled_rows = []
for (model_name, feature_set, alarm_rule, is_primary), group in alarm_folds.groupby(
        ['Model', 'FeatureSet', 'AlarmRule', 'IsPrimary']):
    alarm_pooled_rows.append({
        'Model': model_name, 'FeatureSet': feature_set,
        'AlarmRule': alarm_rule, 'IsPrimary': int(is_primary),
        'Subjects': int(group['Subject'].nunique()), 'Seizures': len(group),
        'SeizuresPredicted': int(group['SeizurePredicted'].sum()),
        'SeizureSensitivity': group['SeizurePredicted'].mean(),
        'FalseAlarms': int(group['FalseAlarms'].sum()),
        'EligibleInterictalHours': group['EligibleInterictalHours'].sum(),
        'FAR_per_h': safe_div(group['FalseAlarms'].sum(),
                              group['EligibleInterictalHours'].sum()),
        'MeanTimeToWarning_min': group.loc[
            group['SeizurePredicted'] == 1, 'TimeToWarning_min'].mean(),
    })
alarm_pooled_summary = pd.DataFrame(alarm_pooled_rows)
alarm_pooled_summary.to_csv(
    OUTPUT_ROOT / 'alarm_sensitivity_pooled_summary.csv', index=False)

# Within each model: feature sets F2-F7 versus F1 using paired subject MCC.
feature_test_frames = []
for model_name in MODEL_ORDER:
    baseline = subject_metrics.loc[
        (subject_metrics['Model'] == model_name) &
        (subject_metrics['FeatureSet'] == 'F1_Handcrafted'),
        ['Subject', 'MCC']].rename(columns={'MCC': 'MCC_F1'})
    rows = []
    for feature_set in FEATURE_SET_DEFINITIONS:
        if feature_set == 'F1_Handcrafted':
            continue
        current = subject_metrics.loc[
            (subject_metrics['Model'] == model_name) &
            (subject_metrics['FeatureSet'] == feature_set),
            ['Subject', 'MCC']].rename(columns={'MCC': 'MCC_Other'})
        paired = baseline.merge(current, on='Subject').dropna()
        try:
            test = wilcoxon(paired['MCC_Other'], paired['MCC_F1'],
                            alternative='two-sided', zero_method='wilcox')
            statistic, pvalue = float(test.statistic), float(test.pvalue)
        except ValueError:
            statistic, pvalue = np.nan, 1.0
        rows.append({
            'Model': model_name,
            'Comparison': f'{feature_set} vs F1_Handcrafted', 'Metric': 'MCC',
            'N_PairedSubjects': len(paired), 'Statistic': statistic,
            'PValue_Uncorrected': pvalue,
        })
    feature_test_frames.append(holm_adjust(pd.DataFrame(rows)))
wilcoxon_features = pd.concat(feature_test_frames, ignore_index=True)
wilcoxon_features.to_csv(
    OUTPUT_ROOT / 'wilcoxon_mcc_features_vs_F1_by_model.csv', index=False)

# Within each feature set: omnibus Friedman test and Holm-adjusted pairwise
# Wilcoxon comparisons among classifiers.
friedman_rows, pairwise_frames = [], []
for feature_set in FEATURE_SET_DEFINITIONS:
    wide = (subject_metrics.loc[subject_metrics['FeatureSet'] == feature_set]
            .pivot(index='Subject', columns='Model', values='MCC')
            .dropna(subset=MODEL_ORDER))
    if len(wide) >= 2:
        test = friedmanchisquare(*(wide[m].to_numpy() for m in MODEL_ORDER))
        statistic, pvalue = float(test.statistic), float(test.pvalue)
    else:
        statistic, pvalue = np.nan, np.nan
    friedman_rows.append({
        'FeatureSet': feature_set, 'Metric': 'MCC', 'N_Subjects': len(wide),
        'Statistic': statistic, 'PValue': pvalue,
    })
    rows = []
    for i, model_a in enumerate(MODEL_ORDER):
        for model_b in MODEL_ORDER[i + 1:]:
            try:
                test = wilcoxon(wide[model_a], wide[model_b],
                                alternative='two-sided', zero_method='wilcox')
                stat, p = float(test.statistic), float(test.pvalue)
            except ValueError:
                stat, p = np.nan, 1.0
            rows.append({
                'FeatureSet': feature_set, 'Comparison': f'{model_a} vs {model_b}',
                'Metric': 'MCC', 'N_PairedSubjects': len(wide),
                'Statistic': stat, 'PValue_Uncorrected': p,
            })
    pairwise_frames.append(holm_adjust(pd.DataFrame(rows)))

friedman_models = pd.DataFrame(friedman_rows)
friedman_models.to_csv(OUTPUT_ROOT / 'friedman_mcc_models.csv', index=False)
wilcoxon_models = pd.concat(pairwise_frames, ignore_index=True)
wilcoxon_models.to_csv(
    OUTPUT_ROOT / 'wilcoxon_mcc_models_by_featureset.csv', index=False)

# Compact ranking table: epoch MCC rank and alarm trade-off are both retained;
# no single composite score is invented.
ranking = cohort_pooled_summary.copy()
ranking['MCC_Rank'] = ranking['MCC'].rank(ascending=False, method='min')
ranking['SeizureSensitivity_Rank'] = ranking['SeizureSensitivity'].rank(
    ascending=False, method='min')
ranking['FAR_Rank'] = ranking['FAR_per_h'].rank(ascending=True, method='min')
ranking = ranking.sort_values(
    ['SeizureSensitivity_Rank', 'FAR_Rank', 'MCC_Rank'], kind='stable')
ranking.to_csv(OUTPUT_ROOT / 'model_feature_ranking.csv', index=False)

timing_summary = (folds.groupby(['Model', 'FeatureSet'], as_index=False)
    .agg(MeanTestInference_ms_per_epoch=('TestInference_ms_per_epoch', 'mean'),
         SDTestInference_ms_per_epoch=('TestInference_ms_per_epoch', 'std'),
         MeanStreamInference_ms_per_epoch=('StreamInference_ms_per_epoch', 'mean'),
         SDStreamInference_ms_per_epoch=('StreamInference_ms_per_epoch', 'std')))
timing_summary.to_csv(OUTPUT_ROOT / 'inference_timing_summary.csv', index=False)

tuning_summary = pd.read_csv(OUTPUT_ROOT / 'tuning_summary.csv')
tuning_frequency = (tuning_summary
    .groupby(['Model', 'FeatureSet', 'TuningMode', 'BestParams'], as_index=False)
    .agg(OuterFolds=('HeldOutSeizure', 'size'),
         MeanBestInnerMCC=('BestInnerMCC', 'mean'),
         MeanFitAndTuneTime_sec=('FitAndTuneTime_sec', 'mean'))
    .sort_values(['Model', 'FeatureSet', 'OuterFolds'],
                 ascending=[True, True, False], kind='stable'))
tuning_frequency.to_csv(
    OUTPUT_ROOT / 'tuning_selection_frequency.csv', index=False)

display(cohort_pooled_summary.sort_values(['Model', 'FeatureSet']))
display(ranking.head(15))
display(friedman_models)
display(wilcoxon_features)
display(wilcoxon_models)
display(timing_summary)
display(tuning_frequency)
display(crossfit_qc)
print('FINAL V10 MULTI-MODEL LOSO QC: PASS')
print('Save /kaggle/working/LOSO_MULTIMODEL_GAPAWARE_V10 as one new Kaggle Dataset.')
